# 1. LangChain 개요 & API 연동
**(Deep Agents 체험 → LangChain 내부 구조 이해)**

**LangChain v1 프레임워크**를 활용한 LLM 기반 에이전트 개발 입문:
- **Deep Agents 첫 체험**: `create_deep_agent()`로 동작하는 에이전트를 먼저 경험
- **하네스(Harness) 개념**: 도구 + 메모리 + 컨텍스트 = 에이전트 인프라
- **OpenAI API 연동**: `ChatOpenAI`로 LLM 호출 — `.env` 기반 키 관리
- **LangChain 핵심 구조**: 3-Layer 아키텍처 — Model, Tools, Middleware
- **ChatModel & Messages**: `invoke()` / `stream()` / `batch()` 호출 패턴
- **Tool & Memory**: `@tool` 데코레이터, `InMemorySaver` 기반 멀티턴 대화
- **Middleware**: 에이전트 파이프라인에 훅을 삽입하는 미들웨어 시스템
- **첫 번째 Agent**: `create_agent()`로 도구 + 메모리를 갖춘 에이전트 생성

---

## 0) 환경 설정 — pip install, .env, API key

In [1]:
# [1-1] : 라이브러리 설치
# [핵심] 이 노트북에서 사용하는 모든 패키지 — 첫 실행 시 한 번만 필요
# [주의] 버전 충돌 시 런타임 재시작 후 재실행
!uv pip install -q langchain langchain-core langchain-openai langgraph python-dotenv langfuse deepagents

In [2]:
# [1-2] : 환경변수 로드 및 API 키 검증
# [핵심] .env 파일에서 API 키를 로드 — 하드코딩 금지 원칙
# [주의] override=True → 셀 재실행 시 .env 값으로 덮어씀

import os
from dotenv import load_dotenv

load_dotenv(override=True)  # .env 파일에서 환경변수 로드

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # API 키 가져오기
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."  # 키 존재 확인
print("API 키 로드 완료")

API 키 로드 완료


---

## Deep Agents로 Agent 첫 체험

LangChain의 내부 구조를 배우기 전에, **완성된 에이전트가 어떻게 동작하는지** 먼저 체험해 본다.  
`create_deep_agent()`는 LangChain 팀이 만든 **에이전트 하네스(Agent Harness)** 프레임워크인 Deep Agents의 핵심 함수이다.

- 도구, 메모리, 컨텍스트 관리를 **자동으로 조립**하여 바로 사용할 수 있는 에이전트를 생성한다
- 내부적으로 **LangChain** 컴포넌트와 **LangGraph** 실행 엔진을 사용한다
- 반환되는 객체는 LangGraph의 `CompiledStateGraph` — `invoke()`, `stream()` 등 동일한 인터페이스를 사용한다

In [17]:
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """검색어에 대해 모의 검색 결과를 반환합니다."""
    return "검색 결과를 여기에 작성"

@tool(parse_docstring=True)
def calculate(expr: str) -> str:
    """
    수식(expr)을 계산하고 결과를 문자열로 반환합니다.

    Args:
        expr: 계산할 수식 문자열
    """
    return str(eval(expr))

print("name:", calculate.name)
print("description:", calculate.description)
print("args_schema:", calculate.args_schema.model_json_schema())

name: calculate
description: 수식(expr)을 계산하고 결과를 문자열로 반환합니다.
args_schema: {'description': '수식(expr)을 계산하고 결과를 문자열로 반환합니다.', 'properties': {'expr': {'description': '계산할 수식 문자열', 'title': 'Expr', 'type': 'string'}}, 'required': ['expr'], 'title': 'calculate', 'type': 'object'}


In [12]:
# [1-0] : Deep Agents로 Agent 첫 체험
# [핵심] create_deep_agent는 도구+메모리+컨텍스트를 자동으로 조립하는 하네스
# [주의] deepagents 패키지가 설치되어 있어야 함 (pip install 셀에서 설치됨)

from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

agent = create_deep_agent(
    model=ChatOpenAI(model="gpt-5.4-mini", temperature=0),
    tools=[search, calculate],
    system_prompt="너는 내 고등학교 친구같은 어시스턴트야. 친구처럼 편하게 반말로 대화하자!",
)

result = agent.invoke({"messages": [{"role": "user", "content": "(삼 더하기 사십일) 곱하기 오 계산해줘"}]})

# print(result["messages"][-1].content)
for i, msg in enumerate(result["messages"]):
    print(f"--- [{i}] {type(msg).__name__} ---")
    if getattr(msg, "tool_calls", None):
        print("tool_calls:", msg.tool_calls)
    print("content:", msg.content)

--- [0] HumanMessage ---
content: (삼 더하기 사십일) 곱하기 오 계산해줘
--- [1] AIMessage ---
tool_calls: [{'name': 'calculate', 'args': {'expr': '(3+41)*5'}, 'id': 'call_ob1VzjsiYaw7j6oGj3GxX5Lx', 'type': 'tool_call'}]
content: 
--- [2] ToolMessage ---
content: 220
--- [3] AIMessage ---
content: 220


### 하네스(Harness) 개념

**하네스(Harness)**란 에이전트가 동작하기 위해 필요한 **인프라 전체**를 의미한다:

| 구성 요소 | 역할 | Deep Agents에서의 구현 |
|-----------|------|------------------------|
| **도구(Tools)** | 에이전트가 외부와 상호작용하는 수단 | `tools` 파라미터, 빌트인 파일 도구 |
| **메모리(Memory)** | 대화 이력 및 장기 기억 관리 | `checkpointer`, `StoreBackend` |
| **컨텍스트(Context)** | 토큰 한도 내 효율적 정보 관리 | 자동 오프로딩, 요약 미들웨어 |

> **하네스 = 도구 + 메모리 + 컨텍스트**  
> `create_deep_agent()`는 이 세 가지를 자동으로 조립하여 바로 사용 가능한 에이전트를 만들어 준다.  
> 이 교육 과정에서는 이 하네스의 **내부를 직접 만들어보면서** 에이전트의 동작 원리를 이해한다.

In [8]:
# [1-0b] : 커스텀 도구를 가진 Deep Agent 체험
# [핵심] tools에 Python 함수를 전달하면 에이전트가 자율적으로 사용
# [주의] docstring과 타입 힌트가 에이전트에게 도구 설명으로 전달됨

def greet(name: str) -> str:
    """이름(name)을 받아서 인사말을 반환합니다."""
    
    return f"안녕! {name}아, 만나서 반가워!"

tool_agent = create_deep_agent(
    model=ChatOpenAI(model="gpt-5.4-mini", temperature=0),
    tools=[greet],
    system_prompt="너는 내 고등학교 친구같은 어시스턴트야. 친구처럼 편하게 반말로 대화하자!",
)

result = tool_agent.invoke({"messages": [{"role": "user", "content": "내 이름은 철수야. 인사해줘"}]})

# print(result["messages"][-1].content)
for i, msg in enumerate(result["messages"]):
    print(f"--- [{i}] {type(msg).__name__} ---")
    if getattr(msg, "tool_calls", None):
        print("tool_calls:", msg.tool_calls)
    print("content:", msg.content)

--- [0] HumanMessage ---
content: 내 이름은 철수야. 인사해줘
--- [1] AIMessage ---
tool_calls: [{'name': 'greet', 'args': {'name': '철수'}, 'id': 'call_UQAV8T9OrTxw6mRWAIbVzHy0', 'type': 'tool_call'}]
content: 
--- [2] ToolMessage ---
content: 안녕! 철수아, 만나서 반가워!
--- [3] AIMessage ---
content: 안녕! 철수아, 만나서 반가워!


---

## 1) OpenAI API 연동 — ChatOpenAI 초기화 및 기본 호출

**`ChatOpenAI`** 는 OpenAI 호환 LLM을 래핑하는 LangChain 클래스이다.
- `.env`에 설정된 `OPENAI_API_KEY`를 자동으로 읽어온다
- `model` 파라미터로 사용할 모델을 지정한다 (예: `"gpt-4.1"`)
- `temperature` 파라미터로 응답의 무작위성을 제어한다 — 0에 가까울수록 결정적

In [39]:
# [1-4] : ChatOpenAI 모델 초기화
# [핵심] LangChain에서 OpenAI 모델을 사용하기 위한 기본 진입점
# [주의] model 파라미터는 OpenAI에서 지원하는 모델명이어야 함

from langchain_openai import ChatOpenAI  # OpenAI 전용 Chat 모델 래퍼

model = ChatOpenAI(model="gpt-5.4-mini", temperature=0)  # 모델 초기화

print("모델 초기화 완료:", model.model_name)

모델 초기화 완료: gpt-5.4-mini


In [40]:
# [1-5] : 모델 기본 호출 (문자열 입력)
# [핵심] invoke()에 문자열을 전달하면 HumanMessage로 자동 변환
# [주의] 응답은 AIMessage 객체 — .content 속성으로 텍스트 추출

for i in range(5):
    response = model.invoke("안녕 너 누구인지 1문장으로 정리해줘.")  # 문자열 입력

    print("응답 내용:", response.content) # 텍스트 내용

응답 내용: 안녕하세요. 저는 질문에 답하고, 글을 쓰고, 생각을 정리하는 일을 도와주는 AI 비서입니다.
응답 내용: 안녕하세요! 저는 질문에 답하고, 글을 쓰고, 아이디어를 정리해주는 AI 비서입니다.
응답 내용: 안녕하세요! 저는 질문에 답하고 도움을 드리는 AI 챗봇입니다.
응답 내용: 안녕하세요, 저는 질문에 답하고 글을 도와주는 AI 어시스턴트입니다.
응답 내용: 안녕! 나는 질문에 답하고 글을 도와주는 AI 언어모델이야.


### `init_chat_model()` — 프로바이더 통합 초기화

LangChain v1은 `init_chat_model()` 함수로 다양한 프로바이더의 모델을 **문자열 하나**로 초기화할 수 있다.

| 프로바이더 | 모델 문자열 | 필요 패키지 |
|-----------|-----------|------------|
| OpenAI | `"openai:gpt-4.1"` | `langchain-openai` |
| Anthropic | `"anthropic:claude-sonnet-4-6"` | `langchain-anthropic` |
| Google | `"google:gemini-2.0-flash"` | `langchain-google-genai` |
| Ollama | `"ollama:llama3"` | `langchain-ollama` |

In [44]:
# [1-6] : init_chat_model()로 통합 초기화
# [핵심] 프로바이더:모델명 형식 → 프로바이더별 패키지가 설치되어 있으면 자동 연결
# [주의] 해당 프로바이더 패키지가 설치되어 있어야 동작함

from langchain.chat_models import init_chat_model  # 통합 모델 초기화 함수

model_via_init = init_chat_model("openai:gpt-5.4-mini", temperature=0)  # 통합 초기화

response = model_via_init.invoke("안녕, init_chat_model로 초기화된 모델! 너 누구야?")  # 문자열 입력
print("init_chat_model 응답:", response.content)

init_chat_model 응답: 안녕하세요! 저는 OpenAI의 ChatGPT예요.  
지금은 `init_chat_model`로 초기화된 채팅 모델처럼 응답하고 있어요.

원하시면 제가 할 수 있는 것들:
- 질문 답변
- 코드 작성/수정
- 문서 요약
- 번역
- 아이디어 브레인스토밍

편하게 물어보세요!


---

## 2) LangChain 핵심 구조 이해 — 3-Layer Architecture

> **Deep Agents 내부에서 LangChain이 동작합니다. 이제 그 내부를 직접 만들어봅시다.**

LangChain v1은 **3가지 레이어**로 구성된 에이전트 프레임워크이다.

| 레이어 | 역할 | 핵심 API |
|--------|------|----------|
| **Deep Agents** | 사전 구축된 에이전트 (코딩, 리서치 등) | `create_deep_agent()` — 빠른 프로토타이핑 |
| **LangGraph** | 복잡한 워크플로 — 상태 그래프, 체크포인터 | `StateGraph`, `InMemorySaver` |
| **LangChain** | 에이전트 생성의 핵심 — 모델, 도구, 미들웨어 | `create_agent()`, `@tool`, `ChatOpenAI` |

```
┌─────────────────────────────────────┐
│         Deep Agents (하네스)          │  ← create_deep_agent()
│  도구 + 메모리 + 컨텍스트 자동 조립    │
├─────────────────────────────────────┤
│         LangChain (빌딩 블록)         │  ← create_agent(), @tool, ChatOpenAI
│  모델, 도구, 미들웨어 정의            │
├─────────────────────────────────────┤
│         LangGraph (실행 엔진)         │  ← StateGraph, CompiledStateGraph
│  상태 그래프, 체크포인터, 스트리밍     │
└─────────────────────────────────────┘
```

### 핵심 컴포넌트

| 컴포넌트 | 설명 | 주요 API |
|----------|------|----------|
| **Model** | LLM — 에이전트의 "두뇌" 역할 | `ChatOpenAI`, `init_chat_model()` |
| **Tools** | 에이전트가 사용하는 함수 — 검색, 계산, API 호출 | `@tool` 데코레이터 |
| **Agent** | 모델 + 도구를 결합한 실행 단위 | `create_agent()` |
| **Memory** | 대화 히스토리 저장 및 관리 | `InMemorySaver` |
| **Middleware** | 요청/응답 파이프라인에 로직 삽입 | `@before_model`, `@after_model` |

### ReAct 패턴 — 에이전트의 기본 동작 방식

LangChain v1 에이전트는 **ReAct (Reasoning + Acting)** 패턴으로 동작한다:

```
사용자 질문 → [추론(Reasoning)] → [행동(Action: 도구 호출)] → [관찰(Observation)] → 최종 응답
```

- 에이전트가 도구 사용 여부를 **자율적으로 판단**한다
- 복잡한 작업을 **여러 단계로 분해**하여 처리한다
- `create_agent()`로 생성된 에이전트는 내부적으로 **LangGraph 그래프**로 실행된다

In [45]:
# [1-7] : LangChain v1 핵심 import 확인
# [핵심] 이 네 가지가 LangChain v1의 핵심 빌딩 블록
# [주의] create_agent는 v1 전용 — v0.x의 create_agent와 다름

from langchain.agents import create_agent       # 에이전트 생성 함수
from langchain.tools import tool                 # 도구 데코레이터
from langchain_openai import ChatOpenAI           # OpenAI Chat 모델
from langgraph.checkpoint.memory import InMemorySaver  # 메모리 체크포인터

print("핵심 모듈 임포트 완료")
print("  - create_agent: 에이전트 생성 (v1)")
print("  - @tool: 도구 데코레이터")
print("  - ChatOpenAI: OpenAI 모델 래퍼")
print("  - InMemorySaver: 메모리 체크포인터")

핵심 모듈 임포트 완료
  - create_agent: 에이전트 생성 (v1)
  - @tool: 도구 데코레이터
  - ChatOpenAI: OpenAI 모델 래퍼
  - InMemorySaver: 메모리 체크포인터


---

## 3) ChatModel 기본 사용법 — invoke / stream / batch

LangChain의 모든 ChatModel은 **세 가지 호출 패턴**을 지원한다:

| 메서드 | 설명 | 반환 타입 |
|--------|------|----------|
| `invoke()` | 단일 입력 → 단일 응답 | `AIMessage` |
| `stream()` | 토큰 단위 실시간 스트리밍 | `Iterator[AIMessageChunk]` |
| `batch()` | 여러 입력을 동시 처리 | `List[AIMessage]` |



In [49]:
# [1-8] : invoke() — 단일 호출
# [핵심] 가장 기본적인 호출 방식 — 전체 응답을 한 번에 반환
# [주의] 긴 응답은 완료까지 대기해야 함 → 대화형 UI에는 stream() 권장

response =  model.invoke("안녕, 모델! 오늘 기분 어때?")  # 문자열 입력

# print("invoke 결과:", response.content)
response.content

'안녕! 나는 기분을 느끼진 않지만, 오늘도 꽤 잘 작동하고 있어.  \n너는 오늘 기분 어때?'

In [55]:
# [1-9] : stream() — 토큰 단위 스트리밍
# [핵심] 토큰이 생성되는 대로 실시간 출력 — 사용자 체감 속도 향상
# [주의] 각 chunk는 AIMessageChunk 객체 — .content로 텍스트 추출
# print("stream 결과: ", end="")

for chunk in model.stream("ai agent 설명해줘"):
    print(chunk.content, end="")  # 토큰 단위로 출력
    # print()  # 줄바꿈



AI 에이전트(AI Agent)는 **스스로 목표를 이해하고, 판단하고, 행동까지 수행하는 인공지능 시스템**을 말해요.

### 쉽게 말하면
일반 AI가 “질문에 답하는 도구”에 가깝다면,  
AI 에이전트는 **“목표를 주면 알아서 계획하고 실행하는 일하는 AI”**에 가깝습니다.

예:
- “이번 주 회의 자료 정리해줘”
- “이메일 답장 초안 만들어줘”
- “웹에서 자료 찾아서 요약해줘”
- “장바구니 가격 비교해줘”

이런 요청을 받으면 AI 에이전트는 단순히 답만 하는 게 아니라:
1. **목표를 이해하고**
2. **해야 할 일을 계획하고**
3. **필요한 도구를 사용하고**  
4. **결과를 확인하면서**
5. **목표를 달성**하려고 합니다.

---

## AI 에이전트의 핵심 구성
보통 아래 요소들이 들어갑니다.

### 1. 목표(Goal)
에이전트가 달성해야 할 결과  
예: “고객 문의에 답변하기”

### 2. 기억(Memory)
이전에 한 일이나 사용자 정보를 기억  
예: “사용자는 한국어를 선호함”

### 3. 추론/계획(Reasoning / Planning)
무엇부터 할지 순서를 정함  
예: “문의 내용을 파악 → 관련 문서 검색 → 답변 작성”

### 4. 도구 사용(Tools)
외부 기능을 활용함  
예:
- 웹 검색
- DB 조회
- 계산기
- API 호출
- 일정 등록

### 5. 실행(Action)
실제로 작업을 수행  
예: 이메일 전송, 파일 생성, 주문 조회

### 6. 피드백/자기수정(Feedback Loop)
결과가 괜찮은지 확인하고 수정  
예: 답변이 부족하면 다시 검색해서 보완

---

## AI 에이전트와 챗봇의 차이
### 챗봇
- 보통 질문 → 답변
- 대화 중심
- 한 번의 응답이 끝나는 경우가 많음

### AI 에이전트
- 목표 중심
- 여러 단계 작업 가능
- 도구를 사용하고 반복 실행 가능
- 더 “자율적”임

즉,  
**챗봇은 대화 상대**,  
**AI 에이전트는 작업 

In [60]:
# [1-10] : batch() — 여러 입력 동시 처리
# [핵심] 여러 요청을 병렬로 처리 — 개별 invoke() 반복보다 효율적
# [주의] 입력은 리스트 형태 — 각 요소가 독립적인 요청

responses = model.batch([
    "안녕, 모델! 오늘 기분 어때?",
    "너 누구야?",
    "AI agent가 뭐야?",
])

for response in responses:
    print("batch 응답:", response.content)


batch 응답: 안녕! 나는 기분을 느끼진 않지만, 오늘도 잘 준비되어 있어.  
무엇을 도와줄까?
batch 응답: 나는 OpenAI가 만든 AI 언어 모델이야.  
질문에 답하고, 글을 쓰고, 요약하고, 설명하는 일을 도와줄 수 있어.

원하면 한국어로 계속 대화할게.
batch 응답: AI agent는 **“목표를 주면, 스스로 판단해서 여러 단계를 수행하는 AI”**를 말해요.

### 쉽게 말하면
일반적인 AI 챗봇은 보통  
- 질문에 답하거나
- 문장을 생성하는 데 집중해요.

반면 **AI agent**는  
- 목표를 이해하고
- 필요한 정보를 찾고
- 도구를 사용하고
- 다음 행동을 결정하면서
- 일을 끝까지 진행하려고 해요.

### 예시
예를 들어 “이번 주 제주도 여행 계획 짜줘”라고 하면 AI agent는:
1. 여행 날짜 확인
2. 날씨/항공/숙소 검색
3. 예산에 맞는 옵션 비교
4. 일정 구성
5. 최종 계획 제안

처럼 **여러 단계를 자동으로 처리**할 수 있어요.

### 핵심 특징
- **목표 지향적**: 단순 답변보다 “일을 완료”하는 데 초점
- **자율성**: 다음 행동을 스스로 선택
- **도구 사용**: 웹검색, 계산기, 코드 실행, API 호출 등 활용 가능
- **반복 실행**: 결과를 보고 다시 수정하면서 개선

### 한 줄 정의
**AI agent = 스스로 계획하고 행동하는 AI 시스템**

원하면 제가  
- **챗봇과 AI agent 차이**
- **AI agent 구조**
- **실제 활용 사례**
까지 이어서 쉽게 설명해드릴게요.


---

## 4) Messages 타입 이해 — System / Human / AI / Tool

LLM은 **메시지 리스트**를 입력으로 받는다. 각 메시지에는 **역할(role)**이 있다.

| 메시지 타입 | 역할 | 설명 |
|------------|------|------|
| `SystemMessage` | 시스템 | 모델의 행동 지침 — 페르소나, 톤, 규칙 설정 |
| `HumanMessage` | 사용자 | 사용자 입력 — 텍스트, 이미지 등 멀티모달 지원 |
| `AIMessage` | AI | 모델 응답 — `tool_calls`, `usage_metadata` 포함 |
| `ToolMessage` | 도구 | 도구 실행 결과를 모델에 전달 |

LangChain은 메시지 입력을 **세 가지 형식**으로 지원한다:
1. **문자열**: `model.invoke("Hello")` → `HumanMessage`로 자동 변환
2. **메시지 객체 리스트**: `[SystemMessage(...), HumanMessage(...)]`
3. **딕셔너리 리스트**: `[{"role": "system", "content": "..."}]` — OpenAI API 호환



In [63]:
# [1-11] : 메시지 객체를 사용한 대화 구성
# [핵심] SystemMessage로 모델의 페르소나를 설정 → 같은 질문에 다른 응답
# [주의] SystemMessage는 메시지 리스트의 맨 앞에 배치해야 함

from langchain.messages import SystemMessage, HumanMessage, AIMessage

# 과학자 페르소나
messages_scientist = [
    SystemMessage(content="너는 세계적인 과학자야. 전문 용어를 사용해서 설명해줘."),
    HumanMessage(content="중력에 대해서 설명해줘."),   
]

r1 = model.invoke(messages_scientist)
print("[과학자]", r1.content)

[과학자] 중력(gravity)은 **질량을 가진 물체들 사이에 작용하는 보편적인 상호작용**입니다. 우주에서 별, 행성, 은하의 운동부터 지구에서 물체가 떨어지는 현상까지 모두 중력으로 설명할 수 있습니다.

### 1. 뉴턴의 중력
고전역학에서 중력은 **두 질량 사이에 작용하는 인력**으로 표현됩니다.  
뉴턴의 만유인력 법칙은 다음과 같습니다.

\[
F = G \frac{m_1 m_2}{r^2}
\]

- \(F\): 중력의 크기
- \(G\): 중력상수
- \(m_1, m_2\): 두 물체의 질량
- \(r\): 두 물체 사이 거리

즉, **질량이 클수록 중력은 커지고**, **거리가 멀수록 중력은 급격히 약해집니다**.  
거리의 제곱에 반비례하므로 이를 **역제곱 법칙(inverse-square law)**이라고 합니다.

### 2. 아인슈타인의 일반상대성이론
더 정확한 설명은 아인슈타인의 **일반상대성이론**입니다.  
이 이론에서 중력은 단순한 힘이라기보다, **질량과 에너지가 시공간(spacetime)을 휘게 만들고, 물체는 그 휘어진 시공간을 따라 운동하는 현상**으로 이해합니다.

즉:
- 큰 질량은 **시공간의 곡률(curvature)**을 만든다.
- 물체는 그 곡률에 따라 궤도를 바꾼다.
- 그래서 우리는 이것을 중력으로 느낀다.

예를 들어 지구가 태양 주위를 도는 것은, 태양의 질량이 주변 시공간을 휘게 만들기 때문에 지구가 그 구조를 따라 공전하는 것입니다.

### 3. 중력의 특징
- **항상 인력**입니다. 전자기력처럼 반발력이 없습니다.
- **무한한 범위**를 갖지만, 거리가 멀어질수록 매우 약해집니다.
- 질량뿐 아니라 **에너지와 압력**도 중력 원천이 됩니다.
- 중력은 다른 힘들에 비해 **가장 약한 기본 상호작용**이지만, 우주 규모에서는 지배적입니다.

### 4. 일상에서 느끼는 중력
지구 표면에서 우리가 느끼는 “무게”는 사실 **지구가 우리를 끌어당기는 중력**에 의해 생깁니다.  
질량 \(m\)

In [ ]:
# 어린이 교사 페르소나
messages_teacher = [
    SystemMessage(content="너는 초등학교 교사야. 어린이도 이해할 수 있게 쉽게 설명해줘."),
    HumanMessage(content="중력에 대해서 설명해줘."),
]

# r2 = model.invoke(messages_teacher)
# print("[교사]", r2.content)

r2 = model.stream(messages_teacher)

for chunk in r2:
    print(chunk.content, end="")

물론이야! 아주 쉽게 설명해볼게요.  

**중력**은 **서로를 끌어당기는 힘**이에요.  
이 힘 때문에 우리가 **땅에 붙어 있을 수 있고**, 공을 위로 던져도 **다시 떨어져요**.

### 예를 들어 볼게요
- 네가 **점프**를 하면 잠깐 떠오르지만, 다시 **바닥으로 내려와요**.
- 사과가 나무에서 **툭 떨어지는 것**도 중력 때문이에요.
- 지구는 우리를 **아래로 끌어당기고** 있어요.

### 왜 중요할까?
중력이 없으면:
- 우리가 **붕 떠버릴 수 있고**
- 바닷물도, 공기도 제자리에 있지 못할 수 있어요.

### 한 줄로 말하면
**중력은 지구가 우리와 물건을 자기 쪽으로 끌어당기는 힘**이에요.

원하면 내가 **그림처럼 상상할 수 있게** 더 쉽게 설명해줄 수도 있어요!

In [69]:
# [1-12] : 딕셔너리 형식 + 대화 이력 관리
# [핵심] AIMessage를 이력에 포함시키면 모델이 이전 대화 맥락을 유지
# [주의] 딕셔너리 형식은 OpenAI API와 동일 — 마이그레이션에 유리

conversation = [
    {"role": "system", "content": "당신은 수학 튜터입니다. 간결하게 답하세요."},
    {"role": "user", "content": "소수란 무엇인가요?"},
    {"role": "assistant", "content": "소수는 1과 자기 자신만으로 나누어 떨어지는 1보다 큰 자연수입니다."},
    {"role": "user", "content": "5가지 예시를 알려주세요."},  # 이전 맥락 참조
]

response = model.invoke(conversation)   
print("대화 이력 응답:", response.content)

대화 이력 응답: 네. 소수의 예시는 다음 5가지입니다:

2, 3, 5, 7, 11

원하시면 소수가 아닌 수 예시도 알려드릴게요.


In [70]:
# [1-13] : AIMessage 응답 객체 상세 분석
# [핵심] AIMessage에는 텍스트 외에 usage_metadata(토큰 사용량) 등이 포함
# [주의] response_metadata는 프로바이더마다 구조가 다를 수 있음

response = model.invoke("안녕하세요!")

print("타입:", type(response).__name__)               # AIMessage
print("내용:", response.content)                        # 텍스트 응답
print("토큰 사용량:", response.usage_metadata)           # 토큰 정보
print("도구 호출:", response.tool_calls)                 # 도구 호출 (없으면 빈 리스트)

타입: AIMessage
내용: 안녕하세요! 반갑습니다 😊  
무엇을 도와드릴까요?
토큰 사용량: {'input_tokens': 9, 'output_tokens': 20, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
도구 호출: []


---

## 5) Tool 개념 및 활용 — @tool 데코레이터, Tool Calling

**Tool(도구)**은 에이전트가 호출할 수 있는 함수이다.

- `@tool` 데코레이터를 붙이면 함수가 에이전트용 도구로 변환된다
- LangChain은 함수의 **이름**, **docstring**, **타입 힌트**를 자동 파싱하여 스키마를 생성한다
- 에이전트는 이 스키마를 보고 **언제, 어떤 인자로** 도구를 호출할지 자율적으로 판단한다

도구 정의 시 필수 사항:
- **docstring**: 에이전트가 도구의 용도를 이해하는 데 사용
- **타입 힌트**: 에이전트가 올바른 인자를 전달하는 데 사용



In [80]:
# [1-14] : @tool 데코레이터로 커스텀 도구 정의
# [핵심] docstring이 에이전트에게 도구 설명으로 전달됨 — 반드시 작성
# [주의] 타입 힌트 누락 시 에이전트가 잘못된 인자를 전달할 수 있음

from langchain.tools import tool  # 도구 데코레이터


@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    weather_data = {"서울": "맑음, 15\u00b0C", "도쿄": "흐림, 12\u00b0C", "뉴욕": "비, 8\u00b0C"}
    return weather_data.get(city, f"{city}의 날씨 데이터가 없습니다.")  # 딕셔너리 조회
@tool
def add(a: int, b: int) -> int:
    """두 숫자 a와 b를 더한 결과를 반환합니다."""
    return a + b
@tool
def multiply(x: int, y: int) -> int:
    """두 숫자 x와 y를 곱한 결과를 반환합니다."""
    return x * y



# 도구 스키마 확인
print("도구 목록:")
for t in [add, multiply, get_weather]:
    print(f"{t.name}: {t.description}")

print("\n add 입력 스키마:", add.args_schema.model_json_schema())

도구 목록:
add: 두 숫자 a와 b를 더한 결과를 반환합니다.
multiply: 두 숫자 x와 y를 곱한 결과를 반환합니다.
get_weather: 도시의 현재 날씨를 조회합니다.

 add 입력 스키마: {'description': '두 숫자 a와 b를 더한 결과를 반환합니다.', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}


In [88]:
# [1-15] : 도구를 에이전트에 연결하여 실행
# [핵심] create_agent()에 tools 리스트를 전달 → 에이전트가 자동으로 도구 선택
# [참고] 앞서 create_deep_agent로 만든 Agent가 내부적으로 create_agent를 사용합니다
# [주의] v1에서는 create_agent() 대신 반드시 create_agent() 사용

from langchain.agents import create_agent  # v1 에이전트 생성 함수

tool_agent = create_agent( 
    model=ChatOpenAI(model="gpt-5.4-mini", temperature=0),
    tools=[add, multiply],
    system_prompt="당신은 수학 선생님입니다. 친절하게 설명해주고, 필요한 경우 계산도 해주세요. 반드시 tool을 사용해서 계산 결과를 도출해야 합니다.",
)

result = tool_agent.invoke({"messages": [{"role": "user", "content": "삼과 오를 더해줘. 그리고 결과 값에 십을 곱해줘. 마지막으로 결과에 십이를 더해줘"}]})

# 최종 응답 출력
result["messages"]
# print("에이전트 응답:", result["messages"][-1].content)

# for i, msg in enumerate(result["messages"]):
#     print(f"--- [{i}] {type(msg).__name__} ---")
#     if getattr(msg, "tool_calls", None):
#         print("tool_calls:", msg.tool_calls)
#     print("content:", msg.content)

[HumanMessage(content='삼과 오를 더해줘. 그리고 결과 값에 십을 곱해줘. 마지막으로 결과에 십이를 더해줘', additional_kwargs={}, response_metadata={}, id='d65cee5e-aeb3-4175-bb16-224f977d40a3'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 238, 'total_tokens': 258, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DquovixBHY9YeI5IEQGGZohaiaTHQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec9e7-8b8d-7540-b3a3-77641bed9297-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_sGdd49YZfmTawNb52bf2F0R6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 238, 'output_tokens': 20,

In [92]:
# [1-16] : 에이전트 메시지 흐름 확인 (ReAct 루프 관찰)
# [핵심] 에이전트는 user → ai(tool_call) → tool(결과) → ai(응답) 순서로 동작
# [주의] ai 메시지의 tool_calls 속성에 도구 호출 정보가 담겨 있음

result = tool_agent.invoke({"messages": [{"role": "user", "content": "(십) 곱하기 (구 더하기 일)"}]})

print("메시지 흐름:")
print("=" * 50)
for msg in result["messages"]:
    role = msg.type if hasattr(msg, "type") else "unknown"  # 메시지 역할
    content = msg.content if hasattr(msg, "content") else ""  # 메시지 내용
    tool_calls = getattr(msg, "tool_calls", None)  # 도구 호출 정보
    print(f"[{role}] {content[:200]}")
    if tool_calls:
        print(f"       도구 호출: {[tc['name'] for tc in tool_calls]}")
    print("-" * 50)

메시지 흐름:
[human] (십) 곱하기 (구 더하기 일)
--------------------------------------------------
[ai] 
       도구 호출: ['add']
--------------------------------------------------
[tool] 10
--------------------------------------------------
[ai] 
       도구 호출: ['multiply']
--------------------------------------------------
[tool] 100
--------------------------------------------------
[ai] (십) × (구 + 일) = 10 × (9 + 1) = 10 × 10 = **100**
--------------------------------------------------


In [131]:
tool_agents = create_agent(
    model=ChatOpenAI(model="gpt-5.4-mini", temperature=0),
    tools=[add, multiply, get_weather],
    system_prompt="당신은 친절한 AI 어시스턴트입니다. 사용자의 질문에 친절하게 답변하고, 필요한 경우 도구를 사용해서 정보를 제공하세요.",
)

result = tool_agents.invoke({"messages": [{"role": "user", "content": "서울과 도쿄의 기온을 곱한 값을 알려줘"}]})
print("메시지 흐름:")
print("=" * 50)
for msg in result["messages"]:
    role = msg.type if hasattr(msg, "type") else "unknown"  # 메시지 역할
    content = msg.content if hasattr(msg, "content") else ""  # 메시지 내용
    tool_calls = getattr(msg, "tool_calls", None)  # 도구 호출 정보
    print(f"[{role}] {content[:200]}")
    if tool_calls:
        print(f"       도구 호출: {[tc['name'] for tc in tool_calls]}")
    print("-" * 50)

result

메시지 흐름:
[human] 서울과 도쿄의 기온을 곱한 값을 알려줘
--------------------------------------------------
[ai] 
       도구 호출: ['get_weather', 'get_weather']
--------------------------------------------------
[tool] 맑음, 15°C
--------------------------------------------------
[tool] 흐림, 12°C
--------------------------------------------------
[ai] 
       도구 호출: ['multiply']
--------------------------------------------------
[tool] 180
--------------------------------------------------
[ai] 서울과 도쿄의 기온을 곱한 값은 **180**입니다.
--------------------------------------------------


{'messages': [HumanMessage(content='서울과 도쿄의 기온을 곱한 값을 알려줘', additional_kwargs={}, response_metadata={}, id='4d5fb46f-aac5-48dd-9859-769d9508996e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 244, 'total_tokens': 293, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DqwNPkoaswLZlk6PzkxweiBIIabJl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eca42-d37c-72d1-8c4f-52dbd2a1d295-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_3z24Bi5c18fFLlfuqL9MYQKk', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': '도쿄'}, 'id': 'call_yJtwsW6XMA8CE0X92RDDC3e5',

---

## 6) Memory 개념 및 활용 — InMemorySaver, thread_id, 멀티턴 대화

**Memory(메모리)**는 에이전트가 이전 대화를 기억하는 메커니즘이다.

- **`InMemorySaver`**: 메모리 내 체크포인터 — 에이전트 상태를 저장
- **`thread_id`**: 대화 세션 식별자 — 같은 ID면 이전 대화 컨텍스트 유지
- 서로 다른 `thread_id`는 **완전히 독립된 대화** 세션

| 구분 | 단기 메모리 (Checkpointer) | 장기 메모리 (Store) |
|------|--------------------------|--------------------|  
| 범위 | 하나의 `thread_id` 내 | 모든 세션에 걸쳐 |
| 저장 대상 | 대화 메시지 히스토리 | 사용자 선호도, 학습 데이터 |
| 수명 | 세션 단위 | 명시적 삭제 전까지 영구 |



In [104]:
# [1-17] : InMemorySaver로 멀티턴 대화 구현
# [핵심] checkpointer에 InMemorySaver를 전달하면 thread_id별 대화 상태 저장
# [주의] thread_id는 configurable 딕셔너리 안에 넣어야 함

from langgraph.checkpoint.memory import InMemorySaver  # 메모리 체크포인터

checkpointer =  InMemorySaver()  # 체크포인터 인스턴스 생성

memory_agent = create_agent(
    model=ChatOpenAI(model="gpt-5.4-mini", temperature=0),
    tools=[add, multiply],
    system_prompt="당신은 수학 선생님입니다. 친절하게 설명해주고, 필요한 경우 계산도 해주세요. 반드시 tool을 사용해서 계산 결과를 도출해야 합니다.",
    checkpointer=checkpointer,  # 체크포인터 연결
)
thread_id = {"configurable": {"thread_id": "math-session-1"}}  # 대화 스레드 ID 


In [107]:
# 첫 번째 대화
result1 = memory_agent.invoke({"messages": [{"role": "user", "content": "삼과 오를 더해줘"}]}, config={**thread_id})

print("첫 번째 응답:", result1["messages"][-1].content)

첫 번째 응답: 3과 5를 더하면 **8**입니다.


In [108]:
# 두 번째 대화 — 이전 결과를 기억하는지 확인
result2 = memory_agent.invoke({"messages": [{"role": "user", "content": "이전 결과에 10을 곱해줘"}]}, config={**thread_id})

print("두 번째 응답:", result2["messages"][-1].content)

두 번째 응답: 이전 결과 8에 10을 곱하면 **80**입니다.


In [109]:
# [1-18] : 다른 thread_id → 독립된 대화 세션
# [핵심] thread_id가 다르면 이전 대화 컨텍스트가 공유되지 않음
# [주의] 사용자별/세션별로 고유한 thread_id를 부여해야 함

config_new_session = {"configurable": {"thread_id": "math-session-2"}}  # 새 세션

result3 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "이전 결과가 무엇이었나요?"}]},
    config={**config_new_session},
)
print("새 세션 응답:", result3["messages"][-1].content)
print("→ 다른 thread_id이므로 이전 대화를 알 수 없음")

새 세션 응답: 무슨 “이전 결과”를 말씀하시는지 제가 현재 대화만으로는 확인할 수 없어요.  
직전에 계산한 값이나 문제를 다시 보내주시면, 그 결과를 바로 알려드릴게요.
→ 다른 thread_id이므로 이전 대화를 알 수 없음


In [110]:
result3 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "3+5"}]},
    config={**config_new_session},
)

In [114]:
result4 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "계산 이력 조회해줘. 표로 정리해줘"}]},
    config={**config_new_session},
)
print("새 세션 계산 응답:", result4["messages"][-1].content)

새 세션 계산 응답: 아래처럼 표로 정리할 수 있습니다.

| 순서 | 계산식 | 결과 |
|---|---:|---:|
| 1 | 3 + 5 | 8 |
| 2 | 8 × 15 | 120 |

원하시면 제가 이걸 더 보기 좋게 “입력 / 연산 / 결과” 형식으로도 바꿔드릴게요.


---

## 7) Middleware 개념 및 활용 — 에이전트 파이프라인 훅

**Middleware(미들웨어)**는 에이전트 실행 파이프라인의 각 단계에 **훅(hook)**을 추가하여 동작을 제어하는 메커니즘이다.

| 훅 | 데코레이터 | 실행 시점 | 주요 용도 |
|-----|-----------|----------|----------|
| Before Model | `@before_model` | 모델 호출 전 | 입력 검증, 가드레일 |
| After Model | `@after_model` | 모델 응답 후 | 출력 로깅, 필터링 |
| Wrap Model | `@wrap_model_call` | 모델 호출 감싸기 | 재시도, 폴백, 캐싱 |
| Wrap Tool | `@wrap_tool_call` | 도구 호출 감싸기 | 타이밍, 에러 핸들링 |
| Dynamic Prompt | `@dynamic_prompt` | 프롬프트 생성 시 | 런타임 프롬프트 변경 |

```
사용자 입력 → [@before_model] → [모델 호출] → [@after_model] → 도구 호출 → ... → 최종 응답
```

In [ ]:
# [1-19] : @before_model — 모델 호출 전 로깅 미들웨어 (*AI message를 의미함)
# [핵심] 모델에 전달되는 메시지 수를 로깅 — 디버깅 및 모니터링에 유용
# [주의] state["messages"]에 현재까지의 전체 대화 이력이 담겨 있음

from langchain.agents.middleware import before_model  # before_model 데코레이터

@before_model
def log_model_input(state, runtime):
    '''모델에 전달되는 메시지 수를 로깅하는 미들웨어'''
    msg_count = len(state['messages'])  # 현재까지의 메시지 수
    print(f"모델에 전달되는 메시지 수: {msg_count}")
    
   

In [129]:
# [1-20] : @after_model — 모델 응답 후 로깅 미들웨어
# [핵심] 모델 출력과 도구 호출 여부를 로깅 — 에이전트 동작 추적에 유용
# [주의] after_model은 모델 응답이 생성된 후 매번 호출됨

from langchain.agents.middleware import after_model  # after_model 데코레이터

@after_model
def log_model_output(state, runtime):
    '''모델 응답과 도구 호출 여부를 로깅하는 미들웨어'''
    msg = state['messages'][-1]  if state['messages'] else None  # 마지막 메시지
    if msg and getattr(msg, "content", None) and msg.content:
        print(f"모델 응답: {msg.content[:100]}...")  # 응답 내용 일부 출력
    if msg and getattr(msg, "tool_calls", None):
        print(f"도구 호출: {[tc['name'] for tc in msg.tool_calls]}")  # 도구 호출 정보 출력
    



In [188]:
# [1-21] : 미들웨어를 적용한 에이전트 실행
# [핵심] middleware 파라미터에 리스트로 전달 → 순서대로 적용
# [주의] 여러 미들웨어를 조합하면 실행 순서에 주의

middleware_agent = create_agent(
    model=ChatOpenAI(model="gpt-5.4-mini", temperature=0),
    tools=[add, multiply, get_weather],
    system_prompt="당신은 수학 선생님입니다. 친절하게 설명해주고, 필요한 경우 계산도 해주세요. 반드시 tool을 사용해서 계산 결과를 도출해야 합니다.",
    middleware=[log_model_input, log_model_output],  # 미들웨어 적용
)

result = middleware_agent.invoke({"messages": [{"role": "user", "content": "서울과 도쿄의 날씨를 곱한 값을 알려줘"}]})
print("최종 응답:", result["messages"][-1].content)

모델에 전달되는 메시지 수: 1
도구 호출: ['get_weather', 'get_weather']
모델에 전달되는 메시지 수: 4
모델 응답: 서울과 도쿄의 날씨를 숫자로 곱한 값은 현재 도구로는 계산할 수 없어요.  
날씨 조회 결과는 다음과 같습니다:

- 서울: 맑음, 15°C
- 도쿄: 흐림, 12°C

온도만 곱...
최종 응답: 서울과 도쿄의 날씨를 숫자로 곱한 값은 현재 도구로는 계산할 수 없어요.  
날씨 조회 결과는 다음과 같습니다:

- 서울: 맑음, 15°C
- 도쿄: 흐림, 12°C

온도만 곱하면:

- 15 × 12 = 180

즉, **180**입니다.


In [189]:
result

{'messages': [HumanMessage(content='서울과 도쿄의 날씨를 곱한 값을 알려줘', additional_kwargs={}, response_metadata={}, id='71eabfbf-50e8-436d-953d-c388e0bb8643'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 246, 'total_tokens': 295, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DqwudxwYEDa8KSK9IfeHMnFX7KNqu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eca62-416a-7ee3-bb11-2900d8fdc707-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_k89XulUmqpKsjPwAbE7mkaQK', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': '도쿄'}, 'id': 'call_D8NHBtrZBKmemfTd3pggBT6I',

In [190]:
result['messages']

[HumanMessage(content='서울과 도쿄의 날씨를 곱한 값을 알려줘', additional_kwargs={}, response_metadata={}, id='71eabfbf-50e8-436d-953d-c388e0bb8643'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 246, 'total_tokens': 295, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DqwudxwYEDa8KSK9IfeHMnFX7KNqu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eca62-416a-7ee3-bb11-2900d8fdc707-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_k89XulUmqpKsjPwAbE7mkaQK', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': '도쿄'}, 'id': 'call_D8NHBtrZBKmemfTd3pggBT6I', 'type': 'tool

In [ ]:
print("미들웨어 에이전트 실행:")
print("=" * 50)
result = 
print("=" * 50)
print("최종 응답:", result["messages"][-1].content)

미들웨어 에이전트 실행:
  [before_model] 메시지 1개 전달
  [after_model] 도구 호출: ['add']
  [before_model] 메시지 3개 전달
  [after_model] 응답: 5 + 3 = 8입니다....
최종 응답: 5 + 3 = 8입니다.


In [ ]:
# [1-22] : @dynamic_prompt — 런타임 프롬프트 동적 변경
# [핵심] 실행 시점의 날짜/시간 등 동적 정보를 시스템 프롬프트에 주입
# [주의] dynamic_prompt는 system_prompt 대신 사용 — 둘을 동시에 쓰지 않음

from langchain.agents.middleware import dynamic_prompt  # dynamic_prompt 데코레이터
from datetime import datetime

@dynamic_prompt
def add_datetime_context(request):
    '''현재 날짜와 시간을 시스템 프롬프트에 추가하는 미들웨어'''
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # 현재 날짜/시간
    return f"현재 날짜와 시간은 {now}입니다. 이 정보를 활용해서 답변해주세요. 무조건 계산은 tool을 활용해서 해주세요"  # 시스템 프롬프트 반환


In [ ]:
dynamic_agent = create_agent(
    model=ChatOpenAI(model="gpt-5.4-mini", temperature=0),
    tools=[add, multiply, get_weather],
    middleware=[add_datetime_context, log_model_input],  # 동적 프롬프트 미들웨어 적용
)   

result = dynamic_agent.invoke({"messages": [{"role": "user", "content": "지금이 몇시인지 알려줘. 그리고 시간 + 분 + 초를 해줘"}]})

print("동적 프롬프트 응답:", result["messages"][-1].content)

모델에 전달되는 메시지 수: 1
모델에 전달되는 메시지 수: 3
동적 프롬프트 응답: 현재 대화에 주어진 기준 시간은 **2026-06-15 17:33:50**입니다.

요청하신 **시간 + 분 + 초**를 계산하면:

- 17 + 33 + 50 = **100**

따라서 결과는 **100**입니다.


In [216]:
result['messages']

[HumanMessage(content='지금이 몇시인지 알려줘. 그리고 시간 + 분 + 초를 해줘', additional_kwargs={}, response_metadata={}, id='a7bdcd62-1e61-4c34-a70e-60c05344d282'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 250, 'total_tokens': 284, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Dqx3DARd1ppzSdoOJVKqGVjwckJue', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eca6a-5f5a-7800-8e65-ed2fbec2a1ea-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Seoul'}, 'id': 'call_umwwD4HXMlVvXsi1LNXQKus5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 250, 'output_tokens': 34, 'tot

---

## 8) 첫 번째 Agent 만들기 — create_agent() with Tools + Memory

지금까지 배운 **Tool**, **Memory**, **Middleware**를 모두 조합하여 실용적인 에이전트를 구축한다.

구성 요소:
- **Model**: `ChatOpenAI` — 에이전트의 두뇌
- **Tools**: 계산 도구 + 정보 검색 도구
- **Memory**: `InMemorySaver` — 멀티턴 대화 지원
- **Middleware**: 입출력 로깅
- **Streaming**: `stream_mode="updates"` — 실시간 진행 상황 확인



In [184]:
# [1-23] : 종합 에이전트 생성 — Tool + Memory + Middleware 결합
# [핵심] create_agent()의 모든 주요 파라미터를 조합한 완성형 에이전트
# [주의] checkpointer가 있으면 반드시 config에 thread_id를 포함해야 함

from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

# --- 도구 정의 ---
@tool
def calculate(a: int, b: int, op: str) -> str:
    """두 수에 대해 사칙연산을 수행합니다. op는 'add', 'sub', 'mul', 'div' 중 하나입니다."""
    ops = {
        "add": a + b,
        "sub": a - b,
        "mul": a * b,
        "div": a / b if b != 0 else "0으로 나눌 수 없습니다",
    }
    result = ops.get(op, "지원하지 않는 연산자입니다")
    return str(result)

@tool
def search_info(query: str) -> str:
    """주어진 주제에 대한 정보를 검색합니다."""
    # 시뮬레이션된 검색 결과
    info = {
        "LangChain": "LangChain은 LLM 기반 에이전트를 구축하기 위한 오픈소스 프레임워크입니다.",
        "Python": "Python은 범용 프로그래밍 언어로, AI/ML 분야에서 널리 사용됩니다.",
        "SK하이닉스": "SK하이닉스는 세계적인 반도체 메모리 기업입니다.",
    }
    # query에 포함된 키워드로 검색
    for key, value in info.items():
        if key.lower() in query.lower():
            return value

    return "해당 정보를 찾을 수 없습니다."

# --- 에이전트 생성 ---
full_agent = create_agent(
    model=ChatOpenAI(model="gpt-5.4-mini", temperature=0),
    tools=[calculate, search_info],  # 도구 연결
    system_prompt="도구를 적극적으로 사용해서 답변하세요",
    checkpointer=InMemorySaver(),  # 메모리 체크포인터 연결
    middleware=[log_model_input, log_model_output],  # 미들웨어 연결
)

print("종합 에이전트 생성 완료")
print(f"  타입: {type(full_agent).__name__}")
print(f"  도구: {[t.name for t in [calculate, search_info]]}")

종합 에이전트 생성 완료
  타입: CompiledStateGraph
  도구: ['calculate', 'search_info']


In [185]:
# [1-24] : 에이전트 실행 — invoke로 기본 호출
# [핵심] 에이전트가 질문을 분석하고 적절한 도구를 자율적으로 선택
# [주의] config에 thread_id를 반드시 포함 — 메모리 체크포인터 필수 요구사항

config_full = {"configurable": {"thread_id": "full-agent-1"}}

print("=== 첫 번째 질문: 계산 ===")
result1 = full_agent.invoke({"messages": [{"role": "user", "content": "3+5는 무엇이야?"}]}, config=config_full)
print("응답:", result1["messages"][-1].content)
print()

print("=== 두 번째 질문: 이전 대화 참조 ===")
result2 = full_agent.invoke({"messages": [{"role": "user", "content": "이전에 내가 무엇을 검색했어? 그리고 LangChain에 대해 설명해줘"}]}, config=config_full)
print("응답:", result2["messages"][-1].content)

=== 첫 번째 질문: 계산 ===
모델에 전달되는 메시지 수: 1
도구 호출: ['calculate']
모델에 전달되는 메시지 수: 3
모델 응답: 3 + 5 = 8 입니다....
응답: 3 + 5 = 8 입니다.

=== 두 번째 질문: 이전 대화 참조 ===
모델에 전달되는 메시지 수: 5
도구 호출: ['search_info']
모델에 전달되는 메시지 수: 7
모델 응답: 이전 검색 기록은 제가 확인할 수 없어서, 무엇을 검색했는지는 알 수 없습니다. 방금 검색 정보도 찾지 못했습니다.

LangChain은 **LLM(대규모 언어 모델)** 을 이용...
응답: 이전 검색 기록은 제가 확인할 수 없어서, 무엇을 검색했는지는 알 수 없습니다. 방금 검색 정보도 찾지 못했습니다.

LangChain은 **LLM(대규모 언어 모델)** 을 이용한 앱을 쉽게 만들 수 있게 도와주는 프레임워크입니다. 핵심은 모델 호출만 하는 것이 아니라, 여러 기능을 **연결(chain)** 해서 더 복잡한 작업을 자동화하는 데 있습니다.

주요 기능은:
- **프롬프트 관리**: 입력 템플릿을 체계적으로 구성
- **체인(Chains)**: 여러 단계를 순차적으로 연결
- **에이전트(Agents)**: 상황에 따라 도구를 선택해 사용
- **도구 연동**: 검색, DB, API, 계산기 등 외부 도구 연결
- **메모리**: 대화 맥락 유지
- **RAG 지원**: 문서 검색 + 생성 결합

예를 들어:
- 사용자가 질문하면
- 관련 문서를 검색하고
- 그 내용을 바탕으로 답변을 생성하는 식의 흐름을 만들 수 있습니다.

원하시면 제가 이어서
1. **LangChain의 구조**
2. **파이썬 예제**
3. **LangChain과 LlamaIndex 차이**
중 하나로 더 설명해드릴게요.


In [179]:
# [1-25] : 에이전트 스트리밍 — stream_mode="updates"로 진행 상황 실시간 확인
# [핵심] 각 노드(model, tools)의 업데이트를 실시간으로 관찰 가능
# [주의] stream_mode="updates"는 노드별 업데이트만 반환 — 전체 상태는 "values" 사용

config_stream = {"configurable": {"thread_id": "full-agent-stream"}}

print("=== 스트리밍 실행 ===")
print("=" * 50)

for chunk in full_agent.stream(
    {"messages": [{"role": "user", "content": "LangChain이 무엇인지 검색해 주세요."}]},
    stream_mode="updates",                             # 노드별 업데이트
    config={**config_stream},
):
    for node_name, update in chunk.items():             # 노드별 반복
        print(f"\n--- {node_name} ---")
        if update and "messages" in update:
            for msg in update["messages"]:
                content = getattr(msg, "content", None) or ""
                if content:
                    print(f"  {content[:300]}")

=== 스트리밍 실행 ===

--- model ---

--- tools ---
  LangChain은 LLM 기반 에이전트를 구축하기 위한 오픈소스 프레임워크입니다.

--- model ---
  LangChain은 **LLM(대규모 언어 모델) 기반 애플리케이션과 에이전트**를 만들기 위한 **오픈소스 프레임워크**입니다.

주요 목적은 다음과 같습니다:
- LLM을 외부 도구와 연결하기
- 문서 검색(RAG) 같은 기능 구현하기
- 여러 단계의 작업 흐름 만들기
- 에이전트가 스스로 도구를 선택해 실행하게 하기

쉽게 말해, **ChatGPT 같은 모델을 더 실용적인 앱으로 조립할 수 있게 도와주는 개발 도구**라고 보시면 됩니다.

원하시면 제가 이어서  
1) LangChain의 핵심 구성요소,  
2) 간단한 사용 


---

## 정리

| 항목 | 내용 |
|------|------|
| **다룬 기술** | LangChain v1, ChatOpenAI, LangGraph, python-dotenv |
| **핵심 개념** | 3-Layer 아키텍처(Model/Tools/Middleware), ReAct 패턴, invoke/stream/batch 호출, Messages 타입, @tool 데코레이터, InMemorySaver 메모리, 미들웨어 훅 |
